# Queries with Silver Retail Sales Data

Notebook is imported from Databricks, so the purpose of the inclusion is to show off the queries for Parts C & D rather than a notebook to get the results within this file.

In [0]:
# current Spark version, catalog, and schema
print(f"Spark version: {spark.version}")
print(f"Current catalog: {spark.catalog.currentCatalog()}")

Spark version: 4.1.0
Current catalog: workspace


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Part C

In [0]:
full_orders = spark.read.table("workspace.retail_fresher.silver_full_table")

### Monthly Sales by Category

In [0]:
%sql
SELECT
  order_month,
  category,
  ROUND(SUM(net_sales), 2) AS total_sales
FROM workspace.retail_fresher.silver_full_table
GROUP BY order_month, category
ORDER BY order_month, total_sales DESC
LIMIT 10;

order_month,category,total_sales
2026-01,Fashion,239436.00
2026-01,Grocery,159306.66
2026-01,Home,139416.12
2026-01,Sports,95083.60
2026-01,Electronics,50978.00
2026-02,Grocery,237036.28
2026-02,Fashion,188388.40
2026-02,Home,125620.74
2026-02,Sports,93691.85
2026-02,Electronics,44317.50


### Using AVG, MIN, MAX in Monthly-Category Sales Summary

In [0]:
%sql
SELECT
  order_month,
  category,
  SUM(net_sales) AS total_sales,
  COUNT(order_id) AS order_count,
  AVG(net_sales) AS avg_sales,
  MIN(net_sales) AS min_sales,
  MAX(net_sales) AS max_sales
FROM workspace.retail_fresher.silver_full_table
GROUP BY order_month, category
ORDER BY order_month, category;

order_month,category,total_sales,order_count,avg_sales,min_sales,max_sales
2026-01,Electronics,50978.00,10,5097.800000,772.80,8730.80
2026-01,Fashion,239436.00,15,15962.400000,1084.40,32570.40
2026-01,Grocery,159306.66,11,14482.423636,2156.96,29214.16
2026-01,Home,139416.12,13,10724.316923,591.84,23713.29
2026-01,Sports,95083.60,13,7314.123077,778.05,15898.25
2026-02,Electronics,44317.50,11,4028.863636,0.00,7346.80
2026-02,Fashion,188388.40,13,14491.415385,0.00,30494.40
2026-02,Grocery,237036.28,16,14814.767500,0.00,31947.60
2026-02,Home,125620.74,13,9663.133846,0.00,20443.59
2026-02,Sports,93691.85,12,7807.654167,0.00,14747.80


### Latest Customer Profile

In [0]:
latest_customer_profile = (
    full_orders.crossJoin(full_orders.agg(F.max("signup_date").alias("max_signup_date")))
    .filter(F.col("signup_date") == F.col("max_signup_date"))
    .select(
        "customer_id", "customer_name", "email", "city", "state", "region",
        "customer_segment", "signup_date", "date_of_birth", "is_active",
        "loyalty_points", "updated_at",
    )
)
latest_customer_profile.show(n = 10, truncate = False)

+-----------+-------------+------------------------+-----+------+------+----------------+-----------+-------------+---------+--------------+-------------------+
|customer_id|customer_name|email                   |city |state |region|customer_segment|signup_date|date_of_birth|is_active|loyalty_points|updated_at         |
+-----------+-------------+------------------------+-----+------+------+----------------+-----------+-------------+---------+--------------+-------------------+
|C0409      |Meera Ram    |meera.ram409@example.com|Kochi|Kerala|South |Corporate       |2025-06-18 |1970-02-05   |Y        |133           |2026-02-21 11:00:00|
+-----------+-------------+------------------------+-----+------+------+----------------+-----------+-------------+---------+--------------+-------------------+



### Top Products in Each Category

In [0]:
top_cat_products = (
    full_orders.groupBy("category", "product_name")
    .agg(F.round(F.sum("net_sales"), 2).alias("total_net_sales"))
    .withColumn(
        "product_rank",
        F.rank().over(
            Window.partitionBy("category").orderBy(F.desc("total_net_sales"))
        ),
    )
    .orderBy("product_rank", "category")
)
top_cat_products.show(n = 10, truncate = False)

+-----------+-----------------------+---------------+------------+
|category   |product_name           |total_net_sales|product_rank|
+-----------+-----------------------+---------------+------------+
|Electronics|Headset Model 496      |8730.80        |1           |
|Fashion    |Handbag Model 497      |34992.40       |1           |
|Grocery    |Rice Model 453         |31947.60       |1           |
|Home       |Cookware Model 499     |23713.29       |1           |
|Sports     |Cricket Bat Model 500  |16720.00       |1           |
|Electronics|Mouse Model 491        |8644.30        |2           |
|Fashion    |Shoes Model 492        |34646.40       |2           |
|Grocery    |Snacks Model 498       |29802.36       |2           |
|Home       |Lamp Model 489         |23246.19       |2           |
|Sports     |Tennis Racket Model 495|16555.65       |2           |
+-----------+-----------------------+---------------+------------+
only showing top 10 rows


### Top Customers by State

In [0]:
top_customers_by_state = (
    full_orders.groupBy("state", "customer_id", "customer_name")
    .agg(F.sum("net_amount").alias("total_net_amount"))
    .withColumn(
        "rank",
        F.dense_rank().over(
            Window.partitionBy("state").orderBy(F.desc("total_net_amount"))
        ),
    )
    .select("state", "customer_id", "customer_name", "rank")
    .orderBy("state", "rank")
)
top_customers_by_state.show(n = 10, truncate = False)

+-----+-----------+-------------+----+
|state|customer_id|customer_name|rank|
+-----+-----------+-------------+----+
|Delhi|C0468      |Naveen Patel |1   |
|Delhi|C0178      |Vikram Menon |2   |
|Delhi|C0378      |Vikram Rao   |3   |
|Delhi|C0258      |Vikram Menon |4   |
|Delhi|C0048      |Aarav Kumar  |5   |
|Delhi|C0248      |Aarav Verma  |6   |
|Delhi|C0128      |Aarav Kumar  |7   |
|Delhi|C0328      |Aarav Verma  |8   |
|Delhi|C0038      |Liam Brown   |9   |
|Delhi|C0118      |Liam Brown   |10  |
+-----+-----------+-------------+----+
only showing top 10 rows


### Running Revenue by Category over Months

In [0]:
monthly = (
    full_orders.groupBy("category", "order_month")
    .agg(F.round(F.sum("net_sales"), 2).alias("monthly_revenue"))
)
running_w = (
    Window.partitionBy("category")
    .orderBy("order_month")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
running_rev_cat = (
    monthly
    .withColumn("running_revenue", F.round(F.sum("monthly_revenue").over(running_w), 2))
    .orderBy("category", "order_month")
)
running_rev_cat.show(n = 10, truncate = False)

+-----------+-----------+---------------+---------------+
|category   |order_month|monthly_revenue|running_revenue|
+-----------+-----------+---------------+---------------+
|Electronics|2026-01    |50978.00       |50978.00       |
|Electronics|2026-02    |44317.50       |95295.50       |
|Electronics|2026-03    |29364.40       |124659.90      |
|Electronics|2026-04    |31786.40       |156446.30      |
|Electronics|2026-05    |49847.80       |206294.10      |
|Electronics|2026-06    |34981.20       |241275.30      |
|Fashion    |2026-01    |239436.00      |239436.00      |
|Fashion    |2026-02    |188388.40      |427824.40      |
|Fashion    |2026-03    |240081.60      |667906.00      |
|Fashion    |2026-04    |182385.10      |850291.10      |
+-----------+-----------+---------------+---------------+
only showing top 10 rows


### Latest Order for Each Customer

In [0]:
w = Window.partitionBy("customer_id").orderBy(F.desc("order_timestamp"))
latest_orders = (
    full_orders.withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .orderBy("customer_id")
)
latest_orders.show(n = 10, truncate = False)

+----------+-----------+--------+-------------------+--------+------------+--------------+------------+----------------------+--------------------+-------------+------------+-----------+------------------+-------------+-------------+--------------------------+-----------+-----------+------+----------------+-----------+-------------+---------+--------------+-------------------+---------------------+-----------+-----------------+----------+----------+---------------+--------------+-----------+--------------+-----------+------------+---------------+----------+---------+---------------+
|product_id|customer_id|order_id|order_timestamp    |quantity|discount_pct|payment_method|order_status|promised_delivery_date|actual_delivery_date|sales_channel|warehouse_id|order_month|late_delivery_flag|delivery_days|customer_name|email                     |city       |state      |region|customer_segment|signup_date|date_of_birth|is_active|loyalty_points|updated_at         |product_name         |category 

### Sales By Month & Category

In [0]:
# monthly category sales
monthly_cat_sales = (
    full_orders
    .select("order_month", "category", "net_sales")
    .groupBy("order_month", "category")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales")
    )
    .orderBy("order_month", F.col("total_sales").desc())
)
monthly_cat_sales.show(truncate = False)

+-----------+-----------+-----------+
|order_month|category   |total_sales|
+-----------+-----------+-----------+
|2026-01    |Fashion    |239436.00  |
|2026-01    |Grocery    |159306.66  |
|2026-01    |Home       |139416.12  |
|2026-01    |Sports     |95083.60   |
|2026-01    |Electronics|50978.00   |
|2026-02    |Grocery    |237036.28  |
|2026-02    |Fashion    |188388.40  |
|2026-02    |Home       |125620.74  |
|2026-02    |Sports     |93691.85   |
|2026-02    |Electronics|44317.50   |
|2026-03    |Fashion    |240081.60  |
|2026-03    |Grocery    |222273.84  |
|2026-03    |Home       |161852.85  |
|2026-03    |Sports     |108117.30  |
|2026-03    |Electronics|29364.40   |
|2026-04    |Fashion    |182385.10  |
|2026-04    |Grocery    |149993.04  |
|2026-04    |Home       |112417.20  |
|2026-04    |Sports     |108333.55  |
|2026-04    |Electronics|31786.40   |
+-----------+-----------+-----------+
only showing top 20 rows


### Sales by City

In [0]:
city_sales = (
    full_orders
    .select("city", "net_sales")
    .groupBy("city")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales")
    )
    .orderBy(F.col("total_sales").desc())
)

city_sales.show(truncate = False)

+-----------+-----------+
|city       |total_sales|
+-----------+-----------+
|Kolkata    |646707.20  |
|Kochi      |604234.70  |
|Bhubaneswar|547022.44  |
|Mumbai     |535789.20  |
|Pune       |474324.24  |
|Chennai    |380696.64  |
|Bengaluru  |305162.15  |
|Delhi      |303587.40  |
|Hyderabad  |159992.80  |
|Jaipur     |72211.40   |
|Unknown    |48056.47   |
+-----------+-----------+



### Sales by Customer

In [0]:
customer_value = (
    full_orders
    .select("customer_id", "customer_name", "net_sales")
    .groupBy("customer_id", "customer_name")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales")
    )
    .orderBy(F.col("total_sales").desc())
)

customer_value.show(truncate = False)

+-----------+-------------+-----------+
|customer_id|customer_name|total_sales|
+-----------+-------------+-----------+
|C0154      |Vikram Rao   |51992.80   |
|C0137      |Meera Ram    |40782.32   |
|C0379      |Sneha Iyer   |34992.40   |
|C0074      |Vikram Rao   |34646.40   |
|C0274      |Vikram Menon |33954.40   |
|C0169      |Meera Ram    |32916.40   |
|C0354      |Vikram Menon |32570.40   |
|C0049      |Meera Singh  |32224.40   |
|C0257      |Meera Singh  |31947.60   |
|C0249      |Meera Ram    |31532.40   |
|C0144      |Aarav Kumar  |30494.40   |
|C0329      |Meera Ram    |30148.40   |
|C0344      |Aarav Verma  |29802.40   |
|C0082      |Vikram Menon |29802.36   |
|C0267      |Sneha Iyer   |29508.26   |
|C0282      |Vikram Rao   |29214.16   |
|C0224      |Aarav Kumar  |29110.40   |
|C0239      |Sophia Shah  |28764.40   |
|C0162      |Vikram Menon |28625.96   |
|C0362      |Vikram Rao   |28037.76   |
+-----------+-------------+-----------+
only showing top 20 rows


## Part D

### Completed Orders

In [0]:
%sql
SELECT *
FROM workspace.retail_fresher.silver_sales_orders
WHERE order_status = 'COMPLETED'
LIMIT 5;

order_id,order_timestamp,customer_id,product_id,quantity,discount_pct,payment_method,order_status,promised_delivery_date,actual_delivery_date,sales_channel,warehouse_id,order_month,late_delivery_flag,delivery_days
O000001,2026-01-08T09:07:00.000Z,C0018,P0030,2,5.00,CARD,COMPLETED,2026-01-12,2026-01-11,MOBILE,WH02,2026-01,N,3
O000002,2026-01-15T10:14:00.000Z,C0035,P0059,3,10.00,CASH,COMPLETED,2026-01-19,2026-01-19,STORE,WH03,2026-01,N,4
O000003,2026-01-22T11:21:00.000Z,C0052,P0088,4,15.00,NET_BANKING,COMPLETED,2026-01-26,2026-01-27,WEB,WH04,2026-01,Y,5
O000004,2026-01-29T12:28:00.000Z,C0069,P0117,5,20.00,UPI,COMPLETED,2026-02-02,2026-02-04,MOBILE,WH05,2026-01,Y,6
O000005,2026-02-05T13:35:00.000Z,C0086,P0146,1,0.00,CARD,COMPLETED,2026-02-09,2026-02-12,STORE,WH01,2026-02,Y,7


### Join the Three Datasets

In [0]:
%sql
SELECT *
FROM workspace.retail_fresher.silver_sales_orders o
JOIN workspace.retail_fresher.silver_products p ON o.product_id = p.product_id
JOIN workspace.retail_fresher.silver_customers c ON o.customer_id = c.customer_id 
LIMIT 5;

order_id,order_timestamp,customer_id,product_id,quantity,discount_pct,payment_method,order_status,promised_delivery_date,actual_delivery_date,sales_channel,warehouse_id,order_month,late_delivery_flag,delivery_days,product_id,product_name,category,subcategory,unit_price,cost_price,supplier_name,stock_quantity,launch_date,product_rating,active_flag,customer_id,customer_name,email,city,state,region,customer_segment,signup_date,date_of_birth,is_active,loyalty_points,updated_at
O000001,2026-01-08T09:07:00.000Z,C0018,P0030,2,5.00,CARD,COMPLETED,2026-01-12,2026-01-11,MOBILE,WH02,2026-01,N,3,P0030,Football Model 030,Sports,Sports Sub-3,669.00,434.85,Nova Supply,140,2022-09-28,3.50,Y,C0018,Vikram Menon,vikram.menon18@example.com,Delhi,Delhi,North,Consumer,2023-07-18,1986-07-19,Y,666,2026-01-03T14:00:00.000Z
O000002,2026-01-15T10:14:00.000Z,C0035,P0059,3,10.00,CASH,COMPLETED,2026-01-19,2026-01-19,STORE,WH03,2026-01,N,4,P0059,Table Model 059,Home,Home Sub-4,1170.70,866.32,Metro Wholesale,17,2023-06-16,3.80,Y,C0035,Sneha Davis,sneha.davis35@example.com,Pune,Maharashtra,West,Home Office,2024-01-21,2003-12-09,Y,1295,2026-01-05T17:00:00.000Z
O000003,2026-01-22T11:21:00.000Z,C0052,P0088,4,15.00,NET_BANKING,COMPLETED,2026-01-26,2026-01-27,WEB,WH04,2026-01,Y,5,P0088,Tea Model 088,Grocery,Grocery Sub-1,1672.40,1053.61,Vertex Goods,144,2024-03-03,4.10,Y,C0052,Naveen Patel,naveen.patel52@example.com,Mumbai,Maharashtra,West,Corporate,2024-07-26,1983-05-26,Y,1924,2026-01-07T20:00:00.000Z
O000004,2026-01-29T12:28:00.000Z,C0069,P0117,5,20.00,UPI,COMPLETED,2026-02-02,2026-02-04,MOBILE,WH05,2026-01,Y,6,P0117,Shoes Model 117,Fashion,Fashion Sub-2,2174.10,1565.35,Global Mart,21,2024-11-19,4.40,Y,C0069,Asha Reddy,asha.reddy69@example.com,Kochi,Kerala,South,Consumer,2025-01-29,2000-10-16,Y,2553,2026-01-09T23:00:00.000Z
O000005,2026-02-05T13:35:00.000Z,C0086,P0146,1,0.00,CARD,COMPLETED,2026-02-09,2026-02-12,STORE,WH01,2026-02,Y,7,P0146,Headset Model 146,Electronics,Electronics Sub-3,2675.80,1632.24,Prime Traders,148,2022-04-25,4.70,Y,C0086,Liam Brown,liam.brown86@example.com,Hyderabad,Telangana,South,Home Office,2023-02-16,1980-03-06,Y,3182,2026-01-12T02:00:00.000Z


### Order Information (using CAST, date functions, & string functions)

In [0]:
%sql
SELECT
  order_id,
  UPPER(TRIM(customer_name)) AS customer_name_clean,
  CONCAT(city, ', ', state) AS location,
  DATE_FORMAT(order_timestamp, 'yyyy-MM') AS order_month,
  DATEDIFF(actual_delivery_date, TO_DATE(order_timestamp)) AS delivery_days_calc,
  CAST(quantity AS INT) AS quantity_int,
  CAST(net_sales AS DECIMAL(12, 2)) AS net_sales_dec
FROM workspace.retail_fresher.silver_full_table
WHERE YEAR(order_timestamp) = 2026
  AND MONTH(order_timestamp) BETWEEN 1 AND 3
LIMIT 20;

order_id,customer_name_clean,location,order_month,delivery_days_calc,quantity_int,net_sales_dec
O000001,VIKRAM MENON,"Delhi, Delhi",2026-01,3,2,1271.10
O000002,SNEHA DAVIS,"Pune, Maharashtra",2026-01,4,3,3160.89
O000003,NAVEEN PATEL,"Mumbai, Maharashtra",2026-01,5,4,5686.16
O000004,ASHA REDDY,"Kochi, Kerala",2026-01,6,5,8696.40
O000005,LIAM BROWN,"Hyderabad, Telangana",2026-02,7,1,2675.80
O000006,SOPHIA NAIR,"Bengaluru, Karnataka",2026-02,2,2,6037.25
O000007,AARAV VERMA,"Chennai, Tamil Nadu",2026-02,3,3,9933.84
O000008,MEERA RAM,"Bhubaneswar, Odisha",2026-02,4,4,14215.06
O000009,VIKRAM RAO,"Kolkata, West Bengal",2026-03,5,5,18730.40
O000011,NAVEEN JOSE,"Delhi, Delhi",2026-03,7,2,10803.40


### Tracking Delivery Quality (using CASE WHEN)

In [0]:
%sql
SELECT
  order_id,
  INITCAP(category) AS category,
  order_status,
  CASE
    WHEN order_status = 'COMPLETED' AND late_delivery_flag = 'Y' THEN 'Completed Late'
    WHEN order_status = 'COMPLETED' AND late_delivery_flag = 'N' THEN 'Completed On Time'
    WHEN order_status = 'RETURNED' THEN 'Returned'
    ELSE 'Other'
  END AS delivery_outcome,
  CASE
    WHEN net_sales >= 1000 THEN 'High'
    WHEN net_sales >= 250 THEN 'Medium'
    ELSE 'Low'
  END AS revenue_band,
  DATE_FORMAT(order_timestamp, 'EEEE') AS order_weekday
FROM workspace.retail_fresher.silver_full_table
ORDER BY order_timestamp
LIMIT 20;

order_id,category,order_status,delivery_outcome,revenue_band,order_weekday
O000463,Grocery,COMPLETED,Completed On Time,High,Friday
O000283,Grocery,COMPLETED,Completed On Time,High,Friday
O000103,Grocery,COMPLETED,Completed On Time,High,Friday
O000206,Sports,COMPLETED,Completed On Time,High,Saturday
O000386,Sports,COMPLETED,Completed On Time,High,Saturday
O000129,Fashion,COMPLETED,Completed Late,High,Sunday
O000309,Fashion,COMPLETED,Completed Late,High,Sunday
O000489,Fashion,COMPLETED,Completed Late,High,Sunday
O000412,Home,COMPLETED,Completed Late,High,Monday
O000232,Home,COMPLETED,Completed Late,High,Monday


### Monthly Sales Summary

In [0]:
%sql
SELECT
  category,
  order_month AS sales_month,
  COUNT(DISTINCT order_id) AS order_count,
  SUM(quantity) AS total_quantity,
  ROUND(SUM(net_sales), 2) AS total_revenue,
  ROUND(AVG(net_sales), 2) AS avg_net_sale,
  ROUND(MIN(net_sales), 2) AS min_net_sale,
  ROUND(MAX(net_sales), 2) AS max_net_sale
FROM workspace.retail_fresher.silver_full_table
WHERE order_status = 'COMPLETED'
GROUP BY category, order_month
ORDER BY category, sales_month
LIMIT 10;

category,sales_month,order_count,total_quantity,total_revenue,avg_net_sale,min_net_sale,max_net_sale
Electronics,2026-01,10,10,50978.00,5097.80,772.80,8730.80
Electronics,2026-02,10,10,44317.50,4431.75,1637.80,7346.80
Electronics,2026-03,8,8,29364.40,3670.55,340.30,8038.80
Electronics,2026-04,8,8,31786.40,3973.30,1032.30,7692.80
Electronics,2026-05,11,11,49847.80,4531.62,599.80,8384.80
Electronics,2026-06,9,9,34981.20,3886.80,1291.80,7952.30
Fashion,2026-01,15,75,239436.00,15962.40,1084.40,32570.40
Fashion,2026-02,11,55,188388.40,17126.22,7658.40,30494.40
Fashion,2026-03,14,70,240081.60,17148.69,2122.40,33954.40
Fashion,2026-04,10,50,182385.10,18238.51,5582.40,31532.40


### Ranking Product Revenue by Category

In [0]:
%sql
SELECT
  category,
  product_id,
  product_name,
  ROUND(SUM(net_sales), 2) AS category_revenue,
  RANK() OVER (
    PARTITION BY category
    ORDER BY SUM(net_sales) DESC
  ) AS product_rank
FROM workspace.retail_fresher.silver_full_table
WHERE order_status = 'COMPLETED'
GROUP BY category, product_id, product_name
ORDER BY category, product_rank
LIMIT 10;

category,product_id,product_name,category_revenue,product_rank
Electronics,P0496,Headset Model 496,8730.80,1
Electronics,P0491,Mouse Model 491,8644.30,2
Electronics,P0486,Keyboard Model 486,8557.80,3
Electronics,P0476,Laptop Model 476,8384.80,4
Electronics,P0471,Headset Model 471,8298.30,5
Electronics,P0456,Monitor Model 456,8038.80,6
Electronics,P0451,Laptop Model 451,7952.30,7
Electronics,P0446,Headset Model 446,7865.80,8
Electronics,P0436,Keyboard Model 436,7692.80,9
Electronics,P0416,Mouse Model 416,7346.80,10


### Top 3 Customers for each State

In [0]:
%sql
WITH customer_sales AS (
  SELECT
    state,
    customer_id,
    customer_name,
    ROUND(SUM(net_sales), 2) AS total_revenue
  FROM workspace.retail_fresher.silver_full_table
  WHERE order_status = 'COMPLETED'
  GROUP BY state, customer_id, customer_name
),
ranked AS (
  SELECT
    state,
    customer_id,
    customer_name,
    total_revenue,
    DENSE_RANK() OVER (
      PARTITION BY state
      ORDER BY total_revenue DESC
    ) AS customer_rank
  FROM customer_sales
)
SELECT *
FROM ranked
WHERE customer_rank <= 3
ORDER BY state, customer_rank;

state,customer_id,customer_name,total_revenue,customer_rank
Delhi,C0468,Naveen Patel,16720.00,1
Delhi,C0178,Vikram Menon,16391.30,2
Delhi,C0258,Vikram Menon,15733.90,3
Karnataka,C0103,Sophia Nair,18977.20,1
Karnataka,C0483,Sneha Davis,16555.65,2
Karnataka,C0073,Meera Ram,15898.25,3
Kerala,C0379,Sneha Iyer,34992.40,1
Kerala,C0169,Meera Ram,32916.40,2
Kerala,C0049,Meera Singh,32224.40,3
Maharashtra,C0082,Vikram Menon,29802.36,1


### Comparison

Analytics gathered from using PySpark functionality were on average much faster. They also had the benefit of reusing the full orders table loaded into memory, while SQL queries are extracting from the retail_fresher schema for each new result.